In [1]:
# Smoke test for fit-cace-SOG.py (ChargeEq long-range renamed to SOG_potential)

import os, sys
import torch

# Ensure working directory is this notebook's folder
try:
    THIS_DIR = os.path.abspath(os.path.dirname(__file__))
except NameError:
    THIS_DIR = os.getcwd()

os.chdir(THIS_DIR)

ROOT_DIR = os.path.abspath(os.path.join(THIS_DIR, ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

import cace
from cace.representations import Cace
from cace.modules import PolynomialCutoff, BesselRBF
from cace.models.atomistic import NeuralNetworkPotential

torch.set_default_dtype(torch.float32)
print("cwd:", os.getcwd())
print("root:", ROOT_DIR)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


cwd: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/fit-4hdnnp-NaCl
root: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq
torch: 2.6.0+cu124
cuda available: False


In [3]:
from cace.data.extxyz_charge import get_dataset_from_extxyz_with_charge
from cace.tools import torch_geometric

cutoff = 5.29
collection = get_dataset_from_extxyz_with_charge(
    train_path=os.path.join(THIS_DIR, "NaCl.xyz"),
    cutoff=cutoff,
    valid_fraction=0.1,
    seed=1,
    atomic_energies={11: -4417.07609365649, 17: -12516.880649933015},
)
print("collection.data_key:", collection.data_key)

# 这里 collection.train 已经是 AtomicData 列表了
train_dataset = collection.train
train_loader = torch_geometric.DataLoader(
    dataset=train_dataset,
    batch_size=2,
    shuffle=True,
    drop_last=True,
)

batch = next(iter(train_loader))
print("batch keys:", sorted(batch.keys))


collection.data_key: {'energy': 'energy', 'forces': 'forces', 'charge': 'charge'}
batch keys: ['atomic_numbers', 'batch', 'cell', 'charge', 'edge_index', 'energy', 'forces', 'positions', 'ptr', 'shifts', 'unit_shifts']


In [4]:
first_item = train_loader.dataset[0]

# 1) Data 本身的 keys（注意不要加括号）
print("first_item keys:", first_item.keys)

# 2) 转成 dict 后的 keys
d = first_item.to_dict()
print("first_item dict keys:", sorted(d.keys()))

# 3) 检查 charge 是否存在
if "charge" in d:
    print("charge shape:", d["charge"].shape, "sum:", float(d["charge"].sum()))
else:
    print("no 'charge' field in first_item.to_dict()")

first_item keys: ['edge_index', 'positions', 'shifts', 'unit_shifts', 'cell', 'atomic_numbers', 'forces', 'energy', 'charge']
first_item dict keys: ['atomic_numbers', 'cell', 'charge', 'edge_index', 'energy', 'forces', 'positions', 'shifts', 'unit_shifts']
charge shape: torch.Size([17]) sum: 1.0


In [6]:
batch = next(iter(train_loader))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [7]:
# Build model (same logic as fit-cace-SOG.py, but no training)

Fourier_node = 18

radial_basis = BesselRBF(cutoff=cutoff, n_rbf=6, trainable=True)
cutoff_fn = PolynomialCutoff(cutoff=cutoff)

cace_representation = Cace(
    zs=[11, 17],
    n_atom_basis=2,
    embed_receiver_nodes=True,
    cutoff=cutoff,
    cutoff_fn=cutoff_fn,
    radial_basis=radial_basis,
    n_radial_basis=8,
    max_l=3,
    max_nu=3,
    num_message_passing=0,
    type_message_passing=["Bchi"],
    args_message_passing={"Bchi": {"shared_channels": False, "shared_l": False}},
    device=device,
    timeit=False,
    forward_features=["atomic_numbers"],  # 加这一行
)

sr_energy = cace.modules.atomwise.Atomwise(
    n_layers=3,
    output_key="SR_energy",
    n_hidden=[32, 16],
    use_batchnorm=False,
    add_linear_nn=True,
)

chi = cace.modules.Atomwise(
    n_layers=3,
    n_hidden=[24, 12],
    n_out=1,
    per_atom_output_key="chi",
    output_key="tot_chi",
    residual=False,
    add_linear_nn=True,
    post_process=torch.square,
    bias=False,
)

charge_eq = cace.modules.ChargeEq(
    dl=1.5,
    sigma=1.0,
    elements=[11, 17],
    feature_key="chi",
    output_key="q_eq",
    ewald_key="SOG_potential",
    system_charge=0.0,
    remove_self_interaction=True,
    aggregation_mode="sum",
    use_sog_kernel=True,
    sog_num_components=Fourier_node,
)

e_add = cace.modules.FeatureAdd(feature_keys=["SR_energy", "SOG_potential"], output_key="CACE_energy")
forces = cace.modules.Forces(energy_key="CACE_energy", forces_key="CACE_forces", calc_stress=False)

model = NeuralNetworkPotential(
    input_modules=None,
    representation=cace_representation,
    output_modules=[sr_energy, chi, charge_eq, e_add, forces],
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable params:", trainable_params)


trainable params: 820


In [10]:
# One forward + backward pass
batch = batch.to(device)
print(sorted(model.model_outputs))
out = model(batch, training=True)
loss = out["CACE_energy"].sum() + out["CACE_forces"].pow(2).mean()


for k in ["SR_energy", "SOG_potential", "CACE_energy", "CACE_forces", "q_eq"]:
    print(k, "in out:", k in out)

print("loss:", float(loss.detach().cpu()))

loss.backward()
print("backward ok")


['CACE_energy', 'CACE_forces', 'SOG_potential', 'SR_energy', 'chi', 'q_eq', 'tot_chi']
SR_energy in out: True
SOG_potential in out: True
CACE_energy in out: True
CACE_forces in out: True
q_eq in out: True
loss: 2.148738384246826
backward ok
